# Laporan UAS: Pemodelan Prediksi Pembelian Pelanggan E-Commerce

**Mata Kuliah:** Data Science<br>
**Oleh       :** Kelompok 1<br>
**Offering   :** TI-A

## I. Pendahuluan
Laporan ini bertujuan untuk <br><br>

## II. Tahapan
Pada bagian ini, fokus pada pembersihan data dan pengelompokan (Exploratory & Segmentation).
1. 

## II. Dataset
Dataset yang digunakan dalam analisis ini adalah E-Commerce Behavior Dataset, yang berisi data perilaku pengguna pada sebuah platform e-commerce. Dataset ini mencakup sekitar 8000 pengguna dengan berbagai atribut yang menggambarkan karakteristik pengguna serta aktivitas mereka saat mengunjungi website. Beberapa variabel yang terdapat dalam dataset antara lain usia pengguna (age), waktu yang dihabiskan di website (time_on_site), jumlah halaman yang dilihat (pages_viewed), jumlah barang yang dimasukkan ke keranjang (cart_items), serta informasi mengenai apakah pengguna melihat diskon (discount_seen) dan apakah pengguna merupakan pelanggan yang kembali (returning_user).
Melalui atribut-atribut tersebut, dataset ini memungkinkan analisis terhadap pola perilaku pelanggan dalam berinteraksi dengan platform e-commerce. Data ini dapat digunakan untuk memahami bagaimana karakteristik dan aktivitas pengguna berhubungan dengan keputusan pembelian (purchase). Dengan memanfaatkan teknik analisis data seperti statistik deskriptif, reduksi dimensi menggunakan PCA, serta metode clustering seperti K-Means, dataset ini dapat membantu mengidentifikasi segmen pelanggan yang memiliki perilaku serupa.
Hasil segmentasi ini diharapkan dapat memberikan wawasan mengenai tipe-tipe pelanggan, seperti pengguna yang hanya menjelajah produk, pengguna yang sensitif terhadap diskon, maupun pelanggan loyal yang memiliki kecenderungan melakukan pembelian lebih tinggi.

Dataset:<br>
[E-Commerce Behavior Dataset](https://www.kaggle.com/datasets/asifxzaman/e-commerce-behavior-dataset8000-users)<br>

In [2]:
# INSIALISASI DATA UNTUK PROSES PREDIKSI

using CSV, DataFrames

# Memanggil data super bersih hasil milestone UTS
println("Memuat dataset final...")
df_uas = CSV.read("../data/df_clean.csv", DataFrame)

# Intip ringkasan data awal untuk memastikan kolomnya lengkap
println("Jumlah baris data siap pakai: ", nrow(df_uas), " baris")
first(df_uas, 5)

Memuat dataset final...
Jumlah baris data siap pakai: 6019 baris


Row,user_id,age,gender,device_type,time_on_site,pages_viewed,previous_purchases,cart_items,discount_seen,ad_clicked,returning_user,avg_session_time,bounce_rate,purchase
,Float64,Float64,String7,String7,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2.0,46.0,Male,Mobile,15.63,9.0,4.0,6.0,1.0,1.0,1.0,19.17,86.73,1.0
2,5.0,38.0,Female,Mobile,26.35,9.0,12.0,4.0,1.0,0.0,0.0,18.15,55.35,1.0
3,6.0,56.0,Female,Desktop,26.6,1.0,12.0,7.0,1.0,1.0,0.0,28.4,13.9,1.0
4,7.0,36.0,Male,Mobile,7.6,11.0,11.0,6.0,1.0,1.0,1.0,4.51,13.12,1.0
5,8.0,40.0,Female,Mobile,26.98,7.0,5.0,2.0,1.0,0.0,0.0,33.34,12.61,1.0


In [4]:
# TAHAP 1: TRAIN-TEST SPLIT (MEMBELAH DATA 80/20)

using Random

# Mengunci pola acakan agar hasil pembagian konsisten setiap kali di-run
Random.seed!(42)

# Ambil data dari file df_clean.csv yang sudah dimuat sebelumnya
df_uas = CSV.read("../data/df_clean.csv", DataFrame)

total_baris = nrow(df_uas)
proporsi_train = 0.8
jumlah_train = floor(Int, proporsi_train * total_baris)

# Membuat urutan indeks yang diacak
indeks_acak = shuffle(1:total_baris)

# Memisahkan indeks menjadi 80% dan 20%
indeks_train = indeks_acak[1:jumlah_train]
indeks_test = indeks_acak[(jumlah_train + 1):end]

# Membelah data berdasarkan indeks acak di atas
data_train = df_uas[indeks_train, :]
data_test = df_uas[indeks_test, :]

println("--- PROSES PEMBELAHAN DATA SELESAI ---")
println("Total data awal     : ", total_baris, " baris")
println("Jumlah Data Train (80%): ", nrow(data_train), " baris (Untuk Belajar)")
println("Jumlah Data Test  (20%): ", nrow(data_test), " baris (Untuk Ujian)")

--- PROSES PEMBELAHAN DATA SELESAI ---
Total data awal     : 6019 baris
Jumlah Data Train (80%): 4815 baris (Untuk Belajar)
Jumlah Data Test  (20%): 1204 baris (Untuk Ujian)


In [5]:
# TAHAP 2: MENYIAPKAN FITUR (X) DAN TARGET (Y)

using DecisionTree

println("--- MEMULAI PEMROSESAN MACHINE LEARNING ---")

# 1. Mendefinisikan Fitur (Soal)
# Kita ambil kolom angka yang memengaruhi keputusan beli
kolom_fitur = [:age, :time_on_site, :pages_viewed, :cart_items, :discount_seen]

# 2. Mengubah Data Train menjadi Matriks X dan Vektor y
X_train = Matrix(data_train[:, kolom_fitur])
y_train = data_train.purchase 

# 3. Mengubah Data Test menjadi Matriks X dan Vektor y
X_test = Matrix(data_test[:, kolom_fitur])
y_test = data_test.purchase 

println("Data siap! Mesin akan belajar dari $(length(y_train)) contoh pelanggan masa lalu.")


--- MEMULAI PEMROSESAN MACHINE LEARNING ---
Data siap! Mesin akan belajar dari 4815 contoh pelanggan masa lalu.


In [6]:
# TAHAP 3: MELATIH MODEL (TRAINING) DENGAN DECISION TREE

println("\nMesin sedang belajar... (Training Model)")

# Menggunakan algoritma Decision Tree Classifier
# Semakin dalam pohon (max_depth), semakin kompleks aturannya
model_pohon = build_tree(y_train, X_train)

println("Model berhasil dilatih! Otak AI sudah terbentuk.")


Mesin sedang belajar... (Training Model)
Model berhasil dilatih! Otak AI sudah terbentuk.


In [7]:
# TAHAP 4: EVALUASI MODEL (Ujian Akhir untuk AI)

println("--- MENGUJI KECERDASAN MODEL ---")

# 1. Menyuruh mesin menebak hasil dari soal ujian (Data Test)
tebakan_mesin = apply_tree(model_pohon, X_test)

# 2. Menghitung Akurasi (Membandingkan tebakan dengan kunci jawaban)
# Tanda titik dua ( .== ) artinya mencocokkan setiap baris satu per satu
jumlah_benar = sum(tebakan_mesin .== y_test)
total_ujian = length(y_test)
akurasi = (jumlah_benar / total_ujian) * 100

# 3. Menampilkan Nilai Rapor
println("Total soal ujian (pelanggan baru) : ", total_ujian, " orang")
println("Tebakan mesin yang benar          : ", jumlah_benar, " orang")
println("Akurasi Prediksi Model            : ", round(akurasi, digits=2), "%")

# --- CONTOH KASUS NYATA PENGGUNAAN MODEL ---
println("\n--- SIMULASI PREDIKSI PELANGGAN BARU ---")
# misal ada pengunjung baru sedang membuka websitemu SEKARANG:
# [Umur, Waktu di Situs, Jumlah Halaman, Barang Keranjang, Lihat Diskon]
pelanggan_baru_A = [20.0, 5.0, 3.0, 1.0, 0.0]  # Mahasiswa yang cuma lihat-lihat sebentar
pelanggan_baru_B = [35.0, 25.0, 12.0, 4.0, 1.0] # Pekerja yang lama browsing dan lihat diskon

tebakan_A = apply_tree(model_pohon, pelanggan_baru_A)
tebakan_B = apply_tree(model_pohon, pelanggan_baru_B)

println("Apakah Pelanggan A akan checkout? (0=Tidak, 1=Ya) : ", tebakan_A)
println("Apakah Pelanggan B akan checkout? (0=Tidak, 1=Ya) : ", tebakan_B)

--- MENGUJI KECERDASAN MODEL ---
Total soal ujian (pelanggan baru) : 1204 orang
Tebakan mesin yang benar          : 1195 orang
Akurasi Prediksi Model            : 99.25%

--- SIMULASI PREDIKSI PELANGGAN BARU ---
Apakah Pelanggan A akan checkout? (0=Tidak, 1=Ya) : 1.0
Apakah Pelanggan B akan checkout? (0=Tidak, 1=Ya) : 1.0
